# Unsupervised Learning: Hierarchical (Agglomerative) Clustering

---

## Introduction

**Hierarchical Clustering** builds a tree of clusters (called a **dendrogram**) by repeatedly merging the two closest clusters until all samples belong to one cluster. The optimal `k` is chosen by cutting the dendrogram at the right height.

### Agglomerative vs. Divisive

| Type | Direction | Approach |
|---|---|---|
| **Agglomerative** (bottom-up) | Each sample starts as its own cluster; merge upward | sklearn default |
| **Divisive** (top-down) | All samples in one cluster; split downward | Less common |

### Linkage Criteria

| Linkage | Merges clusters by | Best for |
|---|---|---|
| `ward` | Minimizes variance increase | Compact, equal clusters |
| `complete` | Maximum distance between clusters | Compact shapes |
| `average` | Average pairwise distance | General use |
| `single` | Minimum distance between clusters | Elongated shapes |

This notebook applies hierarchical clustering to a shopping customer dataset and a synthetic dataset.

---
## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.cluster.hierarchy as shc
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

---
## 2. Dataset — Shopping Customers

In [2]:
customer_data = pd.read_csv('hierarchical-clustering-with-python-and-scikit-learn-shopping-data.csv')

print('Shape:', customer_data.shape)
print('Columns:', customer_data.columns.tolist())
customer_data.head(3)

FileNotFoundError: [Errno 2] No such file or directory: 'hierarchical-clustering-with-python-and-scikit-learn-shopping-data.csv'

In [ ]:
# Use Annual Income and Spending Score
data = customer_data.iloc[:, 3:5].values
feature_names = customer_data.columns[3:5].tolist()

print('Features used:', feature_names)
print('Shape:', data.shape)

plt.figure(figsize=(7, 5))
plt.scatter(data[:, 0], data[:, 1], alpha=0.7, edgecolors='k', linewidths=0.3, s=40)
plt.xlabel(feature_names[0])
plt.ylabel(feature_names[1])
plt.title('Shopping Customers — Annual Income vs. Spending Score')
plt.tight_layout()
plt.show()

---
## 3. Dendrogram — Finding Optimal k

The dendrogram visualizes the full merge hierarchy. The optimal `k` is found by drawing a horizontal line at the largest gap between consecutive merge heights — the number of vertical lines it crosses is `k`.

In [ ]:
plt.figure(figsize=(14, 6))
plt.title('Customer Dendrogram — Ward Linkage', fontsize=13)
dend = shc.dendrogram(shc.linkage(data, method='ward'))
plt.axhline(y=200, color='red', linestyle='--', lw=1.5, label='Cut at height=200 → k=5')
plt.xlabel('Customer Index')
plt.ylabel('Euclidean Distance')
plt.legend()
plt.tight_layout()
plt.show()

---
## 4. Fit AgglomerativeClustering (k=5) and Visualize

In [ ]:
cluster = AgglomerativeClustering(n_clusters=5, linkage='ward')
labels = cluster.fit_predict(data)

sil = silhouette_score(data, labels)
print('Cluster sizes:', pd.Series(labels).value_counts().sort_index().to_dict())
print(f'Silhouette Score: {sil:.4f}')

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(data[:, 0], data[:, 1], c=labels, cmap='rainbow',
            alpha=0.8, edgecolors='k', linewidths=0.3, s=60)
plt.xlabel(feature_names[0])
plt.ylabel(feature_names[1])
plt.title(f'Hierarchical Clustering (k=5) — Silhouette = {sil:.4f}')
plt.colorbar(label='Cluster')
plt.tight_layout()
plt.show()

---
## 5. Comparing Linkage Methods

In [ ]:
linkages = ['ward', 'complete', 'average', 'single']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, link in zip(axes, linkages):
    clf = AgglomerativeClustering(n_clusters=5, linkage=link)
    lbl = clf.fit_predict(data)
    sil = silhouette_score(data, lbl)
    ax.scatter(data[:, 0], data[:, 1], c=lbl, cmap='rainbow',
               alpha=0.7, edgecolors='k', linewidths=0.2, s=25)
    ax.set_title(f'{link}\nSilhouette={sil:.3f}', fontsize=10)
    ax.set_xlabel(feature_names[0])

plt.suptitle('Hierarchical Clustering — Linkage Comparison (k=5)', fontsize=13)
plt.tight_layout()
plt.show()

---
## Conclusion

- The **dendrogram** is hierarchical clustering's key strength — it shows the full merge history and lets you choose `k` without running the algorithm multiple times.
- Cutting at height ~200 with `ward` linkage gives 5 well-separated customer segments, confirmed by a strong silhouette score.
- The **linkage comparison** shows `ward` produces the most compact, equal-sized clusters on this data, while `single` linkage tends to chain points and create imbalanced clusters.
- Hierarchical clustering is computationally expensive ($O(n^2)$ memory) — it works well for small to medium datasets but K-Means or DBSCAN scale better to large data.